<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# Day 2: Classical Models, Evaluation, and Diagnostics

This notebook covers model training, evaluation, hyperparameter tuning, and diagnostics for **binary classification** (Adult Income dataset).
It picks up from Day 1's preprocessing work but includes a self-contained quick-load path.

## Table of Contents

- [0. Scope and Success Criteria](#scope)
- [1. Data Loading and Quick Recap](#data-loading)
- [2. Model-Specific Preprocessing](#preprocessing)
- [3. Parameters vs Hyperparameters](#params)
- [4. Model Zoo](#model-zoo)
  - [4.1 Logistic Regression](#lr)
  - [4.2 SVC](#svc)
  - [4.3 Decision Tree](#dt)
  - [4.4 Random Forest](#rf)
  - [4.5 Gradient Boosting](#gbm)
  - [4.6 MLP](#mlp)
  - [4.7 Tabular Foundation Model (Bonus)](#foundation)
- [5. Splitting Strategies](#splitting)
- [6. Evaluation Metrics](#metrics)
- [7. Overfitting](#overfitting)
- [8. Hyperparameter Tuning](#tuning)
  - [8.4 Optuna (Bonus)](#optuna)
- [9. Model Diagnostics](#diagnostics)
  - [9.1 Feature Importance](#importance)
  - [9.2 Calibration](#calibration)
  - [9.3 Gains and Lift](#gains-lift)
  - [9.4 Error Analysis](#error-analysis)
- [10. Model Comparison Summary](#comparison)
- [11. Regression Mini-Lab (Stretch)](#regression-lab)
- [12. Checklist](#checklist)
- [Acceptance Checks](#acceptance)

## <a id="scope"></a> Section 0 - Scope and Success Criteria

**Scope for Day 2**
- Train and compare models across classical families (linear, SVM, trees, ensembles, MLP).
- Understand parameters vs hyperparameters and model-specific preprocessing needs.
- Splitting strategies, evaluation metrics, and overfitting awareness.
- Cross-validation, hyperparameter tuning, and reproducible sklearn Pipelines.
- Model diagnostics: SHAP, calibration, cumulative gains/lift, error analysis (contrast model).

**Success criteria**
- You can train and compare models from different families on the same preprocessed data.
- You understand which preprocessing each model family requires and why.
- You can set up a reproducible Pipeline with cross-validated hyperparameter tuning.
- You can interpret diagnostic plots and perform structured error analysis.

In [ ]:
import os
from pathlib import Path

# Walk up to the project root (idempotent, safe to re-run)
_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    GroupKFold,
    GroupShuffleSplit,
    TimeSeriesSplit,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
    classification_report,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Optional heavy dependencies
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

print("Environment ready.")
print(f"  xgboost: {HAS_XGB}, lightgbm: {HAS_LGB}, shap: {HAS_SHAP}")

def show(df, n=5):
    display(df.head(n))

## <a id="data-loading"></a> Section 1 - Data Loading and Quick Recap

We reload the Adult Income dataset and re-apply the essential Day 1 preprocessing steps so this notebook is **self-contained**.
The goal is to arrive at clean `X_train`, `X_val`, `X_test`, `y_train`, `y_val`, `y_test` matrices ready for modeling.

In [ ]:
# ── Load raw data ──
adult = pd.read_csv("day1/generated/adult_income_issues.csv")

TARGET_COL = "class"
TARGET_BIN_COL = "target"
SPLIT_COL = "split"
ID_COLS = ["person_id"]

LEAKAGE_COLS = ["post_adjudication_risk_code"]

PROCESS_COLS = [
    "db_source_table", "db_etl_batch_id", "db_row_surrogate_key",
    "db_loaded_at_utc", "dataset_schema_version", "extract_country_code",
    "record_written_at", "dgp_regime",
]

# Binary target
adult[TARGET_BIN_COL] = (
    adult[TARGET_COL].astype(str).str.contains(">50", case=False, regex=False).astype(int)
)

print("Adult shape:", adult.shape)
print("Target positive rate:", round(adult[TARGET_BIN_COL].mean(), 4))

In [ ]:
# ── Split: use the dataset's split column, then carve out validation from train ──
def to_numeric_loose(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip().str.replace("h", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")

train_pool = adult.loc[adult[SPLIT_COL] == "train"].copy()
test_df = adult.loc[adult[SPLIT_COL] == "test"].copy()

# Remove overlapping IDs across splits (synthetic data artifact)
overlap_ids = set(train_pool[ID_COLS[0]]) & set(test_df[ID_COLS[0]])
if overlap_ids:
    train_pool = train_pool.loc[~train_pool[ID_COLS[0]].isin(overlap_ids)].copy()
    print(f"Removed {len(overlap_ids)} overlapping IDs from train pool.")

# Entity-aware train/val split
id_target = train_pool.groupby(ID_COLS[0])[TARGET_BIN_COL].mean().round().astype(int)
id_train, id_val = train_test_split(
    id_target.index.to_numpy(),
    test_size=0.2,
    random_state=SEED,
    stratify=id_target.values,
)
train_df = train_pool.loc[train_pool[ID_COLS[0]].isin(id_train)].copy()
val_df = train_pool.loc[train_pool[ID_COLS[0]].isin(id_val)].copy()

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5} rows={len(frame):5d}  target_rate={frame[TARGET_BIN_COL].mean():.4f}")

assert len(set(train_df[ID_COLS[0]]) & set(val_df[ID_COLS[0]])) == 0
assert len(set(train_df[ID_COLS[0]]) & set(test_df[ID_COLS[0]])) == 0
print("Split integrity check passed.")

In [ ]:
# ── Quick preprocessing (compact Day 1 recap) ──
# Drop non-feature columns
excluded = set(
    [TARGET_COL, TARGET_BIN_COL, SPLIT_COL]
    + ID_COLS + LEAKAGE_COLS + PROCESS_COLS
    + ["case_review_note", "constant_one"]
)
feature_cols = [c for c in adult.columns if c not in excluded]

# Classify into numeric vs categorical
numeric_cols = []
categorical_cols = []
for col in feature_cols:
    s = train_df[col]
    if s.dtype.kind in "biufc":
        numeric_cols.append(col)
        continue
    as_num = to_numeric_loose(s)
    if as_num.notna().mean() >= 0.9:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# ── Build feature matrices ──
def build_features(df, numeric_cols, categorical_cols):
    X = pd.DataFrame(index=df.index)
    for c in numeric_cols:
        X[c] = to_numeric_loose(df[c])
    for c in categorical_cols:
        X[c] = df[c].astype(str).fillna("MISSING")
    return X

X_train_raw = build_features(train_df, numeric_cols, categorical_cols)
X_val_raw = build_features(val_df, numeric_cols, categorical_cols)
X_test_raw = build_features(test_df, numeric_cols, categorical_cols)

y_train = train_df[TARGET_BIN_COL].values
y_val = val_df[TARGET_BIN_COL].values
y_test = test_df[TARGET_BIN_COL].values

print(f"\nFeature matrix shapes: train={X_train_raw.shape}, val={X_val_raw.shape}, test={X_test_raw.shape}")

## <a id="preprocessing"></a> Section 2 - Model-Specific Preprocessing Requirements

Not every model needs the same preprocessing. Understanding **what each model family requires** saves time and avoids mistakes.

| Requirement           | Logistic Reg | SVM    | Decision Tree | Random Forest | GBM (XGB/LGBM) | MLP (NN) |
|-----------------------|:----------:|:------:|:-------------:|:-------------:|:--------------:|:--------:|
| **Scaling**           | Yes        | Yes    | No            | No            | No             | Yes      |
| **One-Hot Encoding**  | Yes        | Yes    | Yes (sklearn)  | Yes (sklearn)  | No (LGBM native) | Yes   |
| **Handle Missing**    | Yes        | Yes    | No (some impl) | No (some impl) | No (native)    | Yes      |
| **Collinearity**      | Yes        | Mild   | No            | No            | No             | Mild     |
| **Feature interactions** | Manual  | Kernel | Native        | Native        | Native         | Native   |

We build **two preprocessing pipelines**: one for scaling-sensitive models, one for tree-based models.

📚 [sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) · [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)

In [ ]:
# ── Preprocessing pipelines for different model families ──

# Numeric pipeline: impute + scale (for scaling-sensitive models)
numeric_transformer_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Numeric pipeline: impute only (for tree-based models)
numeric_transformer_simple = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

# Categorical pipeline: impute + one-hot encode
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# ColumnTransformer for scaling-sensitive models (LogReg, SVM, MLP)
preprocessor_scaled = ColumnTransformer([
    ("num", numeric_transformer_scaled, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# ColumnTransformer for tree-based models (DT, RF, GBM)
preprocessor_tree = ColumnTransformer([
    ("num", numeric_transformer_simple, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# Fit on train, transform all sets
preprocessor_scaled.fit(X_train_raw)
X_train_sc = preprocessor_scaled.transform(X_train_raw)
X_val_sc = preprocessor_scaled.transform(X_val_raw)
X_test_sc = preprocessor_scaled.transform(X_test_raw)

preprocessor_tree.fit(X_train_raw)
X_train_tr = preprocessor_tree.transform(X_train_raw)
X_val_tr = preprocessor_tree.transform(X_val_raw)
X_test_tr = preprocessor_tree.transform(X_test_raw)

feature_names_ohe = preprocessor_tree.get_feature_names_out()

print(f"Scaled features shape:     {X_train_sc.shape}")
print(f"Tree-ready features shape: {X_train_tr.shape}")
print(f"No NaN in train (scaled):  {np.isfinite(X_train_sc).all()}")
print(f"No NaN in train (tree):    {np.isfinite(X_train_tr).all()}")

## <a id="params"></a> Section 3 - Parameters vs Hyperparameters

Two terms that are often confused:

- **Parameters**: values the model **learns from data** during `.fit()`.
  Examples: regression coefficients, tree split thresholds, neural network weights.

- **Hyperparameters**: values the practitioner **sets before training** to control the learning process.
  Examples: regularization strength `C`, tree `max_depth`, learning rate, number of hidden layers.

| Model Family     | Key Parameters (learned)                  | Key Hyperparameters (set by you)                     | Approx. # Parameters* |
|-----------------|-------------------------------------------|------------------------------------------------------|-----------------------|
| Logistic Reg.   | Coefficients (1 per feature)              | `C`, `penalty` (L1/L2), `solver`                    | ~n_features            |
| SVM             | Support vectors, dual coefficients         | `C`, `kernel`, `gamma`, `degree`                     | ~n_support_vectors x n_features |
| Decision Tree   | Split rules, leaf values                   | `max_depth`, `min_samples_split`, `min_samples_leaf` | ~n_nodes               |
| Random Forest   | Ensemble of tree parameters                | `n_estimators`, `max_features`, `max_depth`          | ~n_estimators x n_nodes |
| Gradient Boosting | Sequentially fitted tree parameters       | `learning_rate`, `n_estimators`, `max_depth`         | ~n_estimators x n_nodes |
| MLP (NN)        | Weights and biases per layer               | `hidden_layer_sizes`, `activation`, `learning_rate`  | ~sum(layer_i x layer_i+1) |

*Approximate for the Adult dataset (will compute actual counts below).*

## <a id="model-zoo"></a> Section 4 - Model Zoo: Training Classical Models

We train one model per family on the same data, collect metrics in a registry, and compare.

For each model:
1. Brief intro (what it does, when to use it).
2. Fit on train, predict on val.
3. Collect metrics into `RESULTS_DF`.

In [ ]:
# ── Helper: evaluate a model and store results ──
MODEL_REGISTRY = {}
RESULTS_ROWS = []


def count_parameters(model):
    """Estimate the number of learned parameters for common sklearn models."""
    if hasattr(model, "coef_"):
        return int(np.prod(model.coef_.shape)) + int(np.prod(model.intercept_.shape))
    if hasattr(model, "tree_"):
        return int(model.tree_.node_count)
    if hasattr(model, "estimators_"):
        total = 0
        for est in (model.estimators_ if hasattr(model.estimators_[0], "tree_") else []):
            e = est if hasattr(est, "tree_") else est[0]
            total += e.tree_.node_count
        return total
    if hasattr(model, "coefs_"):  # MLP
        return sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_)
    return None


def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, fit=True):
    """Fit (optionally), predict, compute metrics, store in registry."""
    t0 = time.time()
    if fit:
        model.fit(X_tr, y_tr)
    fit_time = time.time() - t0

    y_pred_tr = model.predict(X_tr)
    y_pred_v = model.predict(X_v)

    # Probabilities (not all models support predict_proba by default)
    if hasattr(model, "predict_proba"):
        y_prob_tr = model.predict_proba(X_tr)[:, 1]
        y_prob_v = model.predict_proba(X_v)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob_tr = model.decision_function(X_tr)
        y_prob_v = model.decision_function(X_v)
    else:
        y_prob_tr = y_pred_tr.astype(float)
        y_prob_v = y_pred_v.astype(float)

    n_params = count_parameters(model)

    row = {
        "model": name,
        "train_acc": accuracy_score(y_tr, y_pred_tr),
        "val_acc": accuracy_score(y_v, y_pred_v),
        "train_f1": f1_score(y_tr, y_pred_tr),
        "val_f1": f1_score(y_v, y_pred_v),
        "train_auc": roc_auc_score(y_tr, y_prob_tr),
        "val_auc": roc_auc_score(y_v, y_prob_v),
        "val_precision": precision_score(y_v, y_pred_v),
        "val_recall": recall_score(y_v, y_pred_v),
        "val_logloss": log_loss(y_v, np.clip(y_prob_v, 1e-15, 1 - 1e-15)) if hasattr(model, "predict_proba") else None,
        "n_params": n_params,
        "fit_time_s": round(fit_time, 2),
    }
    RESULTS_ROWS.append(row)
    MODEL_REGISTRY[name] = {
        "model": model,
        "y_prob_val": y_prob_v,
        "y_pred_val": y_pred_v,
        "y_prob_train": y_prob_tr,
    }

    print(f"[{name}]  train_auc={row['train_auc']:.4f}  val_auc={row['val_auc']:.4f}  "
          f"val_f1={row['val_f1']:.4f}  n_params={n_params}  fit={row['fit_time_s']}s")
    return model

### <a id="lr"></a> 4.1 - Logistic Regression

A linear model that estimates the log-odds of the positive class as a linear combination of features.
Simple, fast, and highly interpretable through its coefficients.
Requires scaling and encoding; sensitive to collinearity (use L1/L2 regularization).

📚 [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) · [sklearn Linear Models Guide](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)

In [ ]:
lr = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000, random_state=SEED)
evaluate_model("LogisticRegression", lr, X_train_sc, y_train, X_val_sc, y_val)

# Inspect coefficients (top features by absolute weight)
coef_df = pd.DataFrame({
    "feature": feature_names_ohe,
    "coef": lr.coef_.ravel(),
    "abs_coef": np.abs(lr.coef_.ravel()),
}).sort_values("abs_coef", ascending=False)

print("\nTop 10 features by |coefficient|:")
show(coef_df, n=10)

### <a id="svc"></a> 4.2 - Support Vector Classifier

Finds the hyperplane that maximizes the margin between classes. The **kernel trick** allows it to learn non-linear boundaries.
Very sensitive to feature scaling. Can be slow on large datasets (training complexity ~O(n^2) to O(n^3)).

📚 [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) · [sklearn SVM Guide](https://scikit-learn.org/stable/modules/svm.html)

In [ ]:
svc = SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=SEED)
evaluate_model("SVC_rbf", svc, X_train_sc, y_train, X_val_sc, y_val)

print(f"\nNumber of support vectors: {svc.n_support_.sum()} "
      f"({svc.n_support_.sum() / len(y_train) * 100:.1f}% of training data)")

### <a id="dt"></a> 4.3 - Decision Tree

Recursively splits the feature space into regions, choosing the split that maximizes information gain (or Gini reduction).
Highly interpretable (you can visualize the tree), but prone to **overfitting** -- especially with no depth limit.

📚 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) · [sklearn Decision Trees Guide](https://scikit-learn.org/stable/modules/tree.html)

In [ ]:
# Shallow tree (controlled)
dt_shallow = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED)
evaluate_model("DecisionTree_d5", dt_shallow, X_train_tr, y_train, X_val_tr, y_val)

# Deep tree (overfitting demo -- note train vs val gap)
dt_deep = DecisionTreeClassifier(max_depth=None, min_samples_leaf=1, random_state=SEED)
evaluate_model("DecisionTree_deep", dt_deep, X_train_tr, y_train, X_val_tr, y_val)

# Visualize the shallow tree
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_shallow, max_depth=3, feature_names=feature_names_ohe,
          class_names=["<=50K", ">50K"], filled=True, rounded=True, ax=ax, fontsize=8)
ax.set_title("Decision Tree (max_depth=5, showing top 3 levels)")
plt.tight_layout()
plt.show()

### <a id="rf"></a> 4.4 - Random Forest

An ensemble of decision trees, each trained on a bootstrap sample with random feature subsets.
Reduces variance compared to a single tree. Less prone to overfitting, but less interpretable.
Feature importance via Mean Decrease in Impurity (MDI).

📚 [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) · Breiman (2001), "Random Forests", *Machine Learning*, 45(1), 5–32

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, max_features="sqrt",
    min_samples_leaf=10, random_state=SEED, n_jobs=-1,
)
evaluate_model("RandomForest", rf, X_train_tr, y_train, X_val_tr, y_val)

# MDI feature importance
imp_rf = pd.DataFrame({
    "feature": feature_names_ohe,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
imp_rf.head(15).plot.barh(x="feature", y="importance", ax=ax, legend=False)
ax.set_title("Random Forest - Top 15 Features (MDI)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### <a id="gbm"></a> 4.5 - Gradient Boosting Models

Builds trees **sequentially**, each correcting the errors of the previous ensemble.
The most common family in industry for tabular data (XGBoost, LightGBM, CatBoost).

Key hyperparameters: `learning_rate` (shrinkage), `n_estimators` (number of boosting rounds), `max_depth`.
LightGBM can handle categorical features natively (no one-hot encoding needed).

📚 [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) · [XGBoost docs](https://xgboost.readthedocs.io/) · [LightGBM docs](https://lightgbm.readthedocs.io/) · Friedman (2001) · Chen & Guestrin (2016), "XGBoost", *KDD*

In [ ]:
# sklearn GradientBoosting (baseline)
gb_sk = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=4,
    min_samples_leaf=20, random_state=SEED,
)
evaluate_model("GBM_sklearn", gb_sk, X_train_tr, y_train, X_val_tr, y_val)

# XGBoost
if HAS_XGB:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=5,
        min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, eval_metric="logloss", verbosity=0,
    )
    evaluate_model("XGBoost", xgb_clf, X_train_tr, y_train, X_val_tr, y_val)

# LightGBM
if HAS_LGB:
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=5,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbose=-1,
    )
    evaluate_model("LightGBM", lgb_clf, X_train_tr, y_train, X_val_tr, y_val)

### <a id="mlp"></a> 4.6 - Neural Network (MLP)

A multi-layer perceptron with fully connected layers. Learns non-linear decision boundaries through stacked layers and activation functions.
sklearn's `MLPClassifier` is simple to use (similar to what AutoML libraries like FLAML/LightAutoML use internally).
Requires scaling; sensitive to initialization and learning rate.

📚 [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) · [sklearn Neural Network Guide](https://scikit-learn.org/stable/modules/neural_networks_supervised.html)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=SEED,
)
evaluate_model("MLP", mlp, X_train_sc, y_train, X_val_sc, y_val)

print(f"\nMLP architecture: input={X_train_sc.shape[1]} -> 128 -> 64 -> 1")
print(f"Loss curve length (epochs trained): {len(mlp.loss_curve_)}")

### <a id="foundation"></a> 4.7 (Bonus) - Tabular Foundation Model

Tabular foundation models are pre-trained on large collections of tabular datasets and can perform zero-shot or few-shot prediction on new tables.
This is an emerging area -- think of it as "GPT for tables."

Below is a placeholder for a `tabicl`-style API call. Uncomment and run if the library is installed.

In [ ]:
# # ── Bonus: Tabular Foundation Model (tabicl) ──
# # pip install tabicl
#
# from tabicl import TabICL
#
# tab_model = TabICL()
# tab_model.fit(X_train_raw[numeric_cols + categorical_cols], y_train)
# y_prob_tab = tab_model.predict_proba(X_val_raw[numeric_cols + categorical_cols])[:, 1]
#
# tab_auc = roc_auc_score(y_val, y_prob_tab)
# print(f"TabICL val AUC: {tab_auc:.4f}")

print("Tabular foundation model section: uncomment above if tabicl is installed.")

In [ ]:
# ── Model Zoo Summary ──
RESULTS_DF = pd.DataFrame(RESULTS_ROWS)

display(
    RESULTS_DF[["model", "train_auc", "val_auc", "val_f1", "val_precision",
                "val_recall", "n_params", "fit_time_s"]]
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
    .style.format({
        "train_auc": "{:.4f}", "val_auc": "{:.4f}", "val_f1": "{:.4f}",
        "val_precision": "{:.4f}", "val_recall": "{:.4f}", "fit_time_s": "{:.2f}",
    })
    .bar(subset=["val_auc"], color="#5fba7d")
)

# Flag overfitting: large train-val AUC gap
RESULTS_DF["overfit_gap"] = RESULTS_DF["train_auc"] - RESULTS_DF["val_auc"]
print("\nOverfit gap (train_auc - val_auc):")
print(RESULTS_DF[["model", "overfit_gap"]].sort_values("overfit_gap", ascending=False).to_string(index=False))

> **Do It Yourself**
>
> 1. Train at least 3 models from different families.
> 2. Compare their val performance in a single table.
> 3. Identify which model overfits the most and hypothesize **why** (hint: model complexity vs data size).

---

## <a id="splitting"></a> Section 5 - Splitting Strategies

The way you split data has a **direct impact on how realistic your evaluation is**.
A wrong split can give you an optimistic estimate that will not hold in production.

We compare: random, stratified, group-aware, time-based, and cross-validation.

📚 [sklearn Cross-validation Guide](https://scikit-learn.org/stable/modules/cross_validation.html) · [GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) · [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)

In [ ]:
# ── 5.1 Random vs Stratified Split ──
# Random split
X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(
    X_train_tr, y_train, test_size=0.2, random_state=SEED,
)
# Stratified split
X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_train_tr, y_train, test_size=0.2, random_state=SEED, stratify=y_train,
)

print("Random split target rates:     "
      f"train={y_r_train.mean():.4f}, test={y_r_test.mean():.4f}")
print("Stratified split target rates:  "
      f"train={y_s_train.mean():.4f}, test={y_s_test.mean():.4f}")
print(f"Full train target rate:         {y_train.mean():.4f}")
print("\nStratified split preserves the target distribution -- critical for imbalanced problems.")

In [ ]:
# ── 5.2 Group Split ──
# When multiple rows share the same entity (person_id), a random split can place
# the same person in both train and test -- this is data leakage.

groups_train = train_df[ID_COLS[0]].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
g_train_idx, g_test_idx = next(gss.split(X_train_tr, y_train, groups=groups_train))

print(f"Group split: train={len(g_train_idx)}, test={len(g_test_idx)}")
overlap = set(groups_train[g_train_idx]) & set(groups_train[g_test_idx])
print(f"Entity overlap between splits: {len(overlap)} (should be 0)")

# ── 5.3 Time-Based Split (demo concept with Retail data) ──
retail = pd.read_csv("day1/generated/retail_panel_issues.csv")
retail["date"] = pd.to_datetime(retail["date"])

cutoff = pd.Timestamp("2023-10-01")
retail_train = retail[retail["date"] < cutoff]
retail_test = retail[retail["date"] >= cutoff]

print(f"\nRetail time split: train dates up to {cutoff.date()}")
print(f"  train rows={len(retail_train)}, test rows={len(retail_test)}")
print(f"  train date range: {retail_train['date'].min().date()} to {retail_train['date'].max().date()}")
print(f"  test  date range: {retail_test['date'].min().date()} to {retail_test['date'].max().date()}")
print("  A random split on time-series data would mix future info into training -- time leakage!")

In [ ]:
# ── 5.4 Cross-Validation comparison ──
# We compare KFold, StratifiedKFold, and GroupKFold on the same model.

# Use a quick model for CV demos
cv_model = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)

# Standard KFold
kf_scores = cross_val_score(cv_model, X_train_sc, y_train, cv=KFold(5, shuffle=True, random_state=SEED), scoring="roc_auc")

# Stratified KFold
skf_scores = cross_val_score(cv_model, X_train_sc, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc")

# GroupKFold (by person_id)
unique_ids = train_df[ID_COLS[0]].values
gkf = GroupKFold(n_splits=5)
gkf_scores = cross_val_score(cv_model, X_train_sc, y_train, cv=gkf, groups=unique_ids, scoring="roc_auc")

cv_compare = pd.DataFrame({
    "method": ["KFold", "StratifiedKFold", "GroupKFold"],
    "mean_auc": [kf_scores.mean(), skf_scores.mean(), gkf_scores.mean()],
    "std_auc": [kf_scores.std(), skf_scores.std(), gkf_scores.std()],
})
show(cv_compare)

print("\nGroupKFold may give lower/different scores because it prevents the same entity")
print("from appearing in both train and validation folds -- more realistic for this data.")

> **Do It Yourself**
>
> 1. Run the same model with random vs stratified vs group CV.
> 2. Compare mean and std of AUC scores.
> 3. Which splitting strategy is most appropriate for the Adult dataset and why?
>
> Hint: the Adult dataset has `person_id` -- does the same person appear multiple times?

---

## <a id="metrics"></a> Section 6 - Evaluation Metrics for Binary Classification

Choosing the right metric is as important as choosing the right model.

- **Threshold-dependent**: accuracy, precision, recall, F1 -- depend on a chosen decision threshold.
- **Threshold-independent**: AUC-ROC, AUC-PR, log-loss -- evaluate the quality of the predicted probabilities.
- **Business-oriented**: cost-weighted metrics, optimal threshold selection.

📚 [sklearn Metrics Guide](https://scikit-learn.org/stable/modules/model_evaluation.html) · [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) · [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

In [ ]:
# ── 6.1 Confusion Matrix and Threshold-Dependent Metrics ──
# Pick the best model from the zoo for detailed metric exploration
best_name = RESULTS_DF.sort_values("val_auc", ascending=False).iloc[0]["model"]
best_entry = MODEL_REGISTRY[best_name]
best_model = best_entry["model"]
y_prob_best = best_entry["y_prob_val"]
y_pred_best = best_entry["y_pred_val"]

print(f"Using model: {best_name}\n")
print(classification_report(y_val, y_pred_best, target_names=["<=50K", ">50K"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_best, display_labels=["<=50K", ">50K"], ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2 Threshold Sensitivity ──
# Show how precision and recall change as we move the decision threshold.

thresholds = np.arange(0.05, 0.96, 0.05)
threshold_metrics = []
for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t):
        continue
    threshold_metrics.append({
        "threshold": t,
        "precision": precision_score(y_val, y_pred_t, zero_division=0),
        "recall": recall_score(y_val, y_pred_t),
        "f1": f1_score(y_val, y_pred_t),
        "accuracy": accuracy_score(y_val, y_pred_t),
    })
tm_df = pd.DataFrame(threshold_metrics)

fig, ax = plt.subplots(figsize=(9, 4))
for col in ["precision", "recall", "f1", "accuracy"]:
    ax.plot(tm_df["threshold"], tm_df[col], label=col, marker=".")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Metric Value")
ax.set_title(f"Threshold vs Metrics - {best_name}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: the default threshold of 0.5 is not always optimal.")
print("In practice, pick the threshold based on business constraints (cost of FP vs FN).")

In [ ]:
# ── 6.3 ROC and Precision-Recall Curves (multi-model overlay) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for name, entry in MODEL_REGISTRY.items():
    prob = entry["y_prob_val"]
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc_val = roc_auc_score(y_val, prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

# Precision-Recall curves
for name, entry in MODEL_REGISTRY.items():
    prob = entry["y_prob_val"]
    prec, rec, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    axes[1].plot(rec, prec, label=f"{name} (AP={ap:.3f})")
baseline_rate = y_val.mean()
axes[1].axhline(y=baseline_rate, color="k", linestyle="--", alpha=0.4, label=f"Baseline ({baseline_rate:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("AUC-ROC summarizes overall discriminative ability.")
print("AUC-PR is more informative when the positive class is rare.")

> **Do It Yourself**
>
> 1. Compute and compare metrics for 2+ models.
> 2. Pick a "best model" under two different business constraints:
>    - Scenario A: minimize false positives (high precision).
>    - Scenario B: maximize recall (catch as many positives as possible).
> 3. What threshold would you use in each scenario?

---

## <a id="overfitting"></a> Section 7 - The Overfitting Problem

Overfitting happens when a model memorizes the training data instead of learning generalizable patterns.
The result: excellent training performance, poor validation/test performance.

We make this tangible by **plotting train vs val metrics as a function of model complexity**.

📚 [sklearn Learning Curves](https://scikit-learn.org/stable/modules/learning_curve.html) · Hastie, Tibshirani & Friedman, *Elements of Statistical Learning*, Ch. 7 (Model Assessment)

In [ ]:
# ── 7.1 Decision Tree: complexity vs performance ──
depths = list(range(1, 30))
dt_results = []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt.fit(X_train_tr, y_train)
    dt_results.append({
        "max_depth": d,
        "train_auc": roc_auc_score(y_train, dt.predict_proba(X_train_tr)[:, 1]),
        "val_auc": roc_auc_score(y_val, dt.predict_proba(X_val_tr)[:, 1]),
        "n_leaves": dt.get_n_leaves(),
    })
dt_res = pd.DataFrame(dt_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(dt_res["max_depth"], dt_res["train_auc"], "o-", label="Train AUC")
axes[0].plot(dt_res["max_depth"], dt_res["val_auc"], "s-", label="Val AUC")
axes[0].set_xlabel("max_depth")
axes[0].set_ylabel("AUC")
axes[0].set_title("Decision Tree: Overfitting as Depth Increases")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Highlight the sweet spot
best_depth = dt_res.loc[dt_res["val_auc"].idxmax(), "max_depth"]
axes[0].axvline(x=best_depth, color="red", linestyle="--", alpha=0.5, label=f"Best depth={best_depth}")
axes[0].legend()

axes[1].plot(dt_res["max_depth"], dt_res["n_leaves"], "o-", color="purple")
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("Number of Leaves")
axes[1].set_title("Model Complexity (# leaves) vs Depth")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best val AUC at max_depth={best_depth}: {dt_res.loc[dt_res['val_auc'].idxmax(), 'val_auc']:.4f}")
print("Beyond this depth, train AUC keeps improving but val AUC degrades -- classic overfitting.")

In [ ]:
# ── 7.2 Gradient Boosting: n_estimators vs performance ──
n_est_range = [10, 25, 50, 100, 200, 300, 500, 800]
gb_results = []
for n in n_est_range:
    gb = GradientBoostingClassifier(
        n_estimators=n, learning_rate=0.1, max_depth=4,
        min_samples_leaf=20, random_state=SEED,
    )
    gb.fit(X_train_tr, y_train)
    gb_results.append({
        "n_estimators": n,
        "train_auc": roc_auc_score(y_train, gb.predict_proba(X_train_tr)[:, 1]),
        "val_auc": roc_auc_score(y_val, gb.predict_proba(X_val_tr)[:, 1]),
    })
gb_res = pd.DataFrame(gb_results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gb_res["n_estimators"], gb_res["train_auc"], "o-", label="Train AUC")
ax.plot(gb_res["n_estimators"], gb_res["val_auc"], "s-", label="Val AUC")
ax.set_xlabel("n_estimators")
ax.set_ylabel("AUC")
ax.set_title("Gradient Boosting: More Trees = More Overfitting Risk")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("With boosting, adding more trees always improves train performance.")
print("But val performance plateaus and can degrade -- early stopping is the standard remedy.")

> **Do It Yourself**
>
> 1. Reproduce the overfitting curve for one model of your choice.
> 2. Identify the "sweet spot" complexity setting.
> 3. What is the bias-variance tradeoff in practical terms?

---

## <a id="tuning"></a> Section 8 - Cross-Validation and Hyperparameter Tuning

Manual hyperparameter search is tedious and error-prone.
sklearn provides `GridSearchCV` and `RandomizedSearchCV` to automate this inside a cross-validation loop.

**Critical**: when tuning is done inside a Pipeline, preprocessing is re-fitted on each CV fold's training data, preventing data leakage.

📚 [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) · [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) · Bergstra & Bengio (2012), "Random Search for Hyper-Parameter Optimization", *JMLR*

In [ ]:
# ── 8.1 Manual loop (to appreciate why GridSearchCV exists) ──
param_grid_manual = {"C": [0.01, 0.1, 1.0, 10.0]}
cv_inner = StratifiedKFold(3, shuffle=True, random_state=SEED)

manual_results = []
for C in param_grid_manual["C"]:
    model = LogisticRegression(C=C, max_iter=500, random_state=SEED)
    scores = cross_val_score(model, X_train_sc, y_train, cv=cv_inner, scoring="roc_auc")
    manual_results.append({"C": C, "mean_auc": scores.mean(), "std_auc": scores.std()})

manual_df = pd.DataFrame(manual_results)
show(manual_df)
print(f"\nBest C: {manual_df.loc[manual_df['mean_auc'].idxmax(), 'C']}")
print("This works for 1 hyperparameter, but quickly becomes unmanageable with more.")

In [ ]:
# ── 8.2 GridSearchCV with a full Pipeline ──
# The Pipeline wraps preprocessing + model together.
# GridSearchCV re-fits the preprocessing on each fold's training data -- no leakage.

pipe_lr = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
])

param_grid_lr = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__penalty": ["l1", "l2"],
    "clf__solver": ["saga"],
}

gs_lr = GridSearchCV(
    pipe_lr,
    param_grid_lr,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="roc_auc",
    n_jobs=-1,
    refit=True,
)
gs_lr.fit(X_train_raw, y_train)

print(f"Best params: {gs_lr.best_params_}")
print(f"Best CV AUC: {gs_lr.best_score_:.4f}")

# Evaluate on held-out val
val_auc_gs = roc_auc_score(y_val, gs_lr.predict_proba(X_val_raw)[:, 1])
print(f"Val AUC (hold-out): {val_auc_gs:.4f}")

In [ ]:
# ── 8.3 RandomizedSearchCV with Random Forest ──
from scipy.stats import randint, uniform

pipe_rf = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
])

param_dist_rf = {
    "clf__n_estimators": randint(100, 500),
    "clf__max_depth": [5, 8, 12, 16, None],
    "clf__min_samples_leaf": randint(5, 50),
    "clf__max_features": ["sqrt", "log2", 0.3, 0.5],
}

rs_rf = RandomizedSearchCV(
    pipe_rf,
    param_dist_rf,
    n_iter=20,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring="roc_auc",
    n_jobs=-1,
    random_state=SEED,
    refit=True,
)
rs_rf.fit(X_train_raw, y_train)

print(f"Best params: {rs_rf.best_params_}")
print(f"Best CV AUC: {rs_rf.best_score_:.4f}")

val_auc_rs = roc_auc_score(y_val, rs_rf.predict_proba(X_val_raw)[:, 1])
print(f"Val AUC (hold-out): {val_auc_rs:.4f}")

# Show top-5 configurations
cv_results = pd.DataFrame(rs_rf.cv_results_).sort_values("rank_test_score")
show(cv_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]], n=5)

### <a id="optuna"></a> 8.4 (Bonus) - Optuna

Bayesian optimization explores the hyperparameter space more efficiently than grid or random search.
Optuna builds a probabilistic model of the objective function and focuses on promising regions.

Uncomment the cell below if `optuna` is installed.

In [ ]:
# # ── Bonus: Optuna hyperparameter search ──
# # pip install optuna
#
# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 500),
#         "max_depth": trial.suggest_int("max_depth", 3, 15),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
#         "subsample": trial.suggest_float("subsample", 0.5, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
#     }
#     model = xgb.XGBClassifier(**params, random_state=SEED, eval_metric="logloss", verbosity=0)
#     cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
#     scores = cross_val_score(model, X_train_tr, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
#     return scores.mean()
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=30)
#
# print(f"Best trial AUC: {study.best_value:.4f}")
# print(f"Best params: {study.best_params}")

print("Optuna section: uncomment above if optuna is installed.")

> **Do It Yourself**
>
> 1. Build a Pipeline for one model family of your choice.
> 2. Run `GridSearchCV` or `RandomizedSearchCV` and report best hyperparameters.
> 3. Compare the CV score to the hold-out val score. Are they close?
>
> Hint: if CV score >> val score, you may be overfitting to the CV folds.

---

## <a id="diagnostics"></a> Section 9 - Model Diagnostics

Going beyond aggregate metrics: **understanding how and where the model succeeds or fails**.

We cover:
- Feature importance (permutation + SHAP)
- Calibration plots
- Cumulative gains and lift curves
- Error analysis with a contrast model

### <a id="importance"></a> 9.1 - Feature Importance

**Permutation importance** (model-agnostic): shuffle one feature at a time, measure how much the metric drops.
**SHAP values**: a principled decomposition of each prediction into feature contributions.

📚 [Permutation Importance](https://scikit-learn.org/stable/modules/permutation_importance.html) · [SHAP documentation](https://shap.readthedocs.io/) · Lundberg & Lee (2017), "A Unified Approach to Interpreting Model Predictions", *NeurIPS*

In [ ]:
# ── 9.1a Permutation Importance ──
from sklearn.inspection import permutation_importance

# Use a tree-based model for diagnostics (fast + good performance)
diag_model_name = "RandomForest"
if diag_model_name not in MODEL_REGISTRY:
    diag_model_name = list(MODEL_REGISTRY.keys())[-1]
diag_model = MODEL_REGISTRY[diag_model_name]["model"]

# Permutation importance on validation set
perm_imp = permutation_importance(
    diag_model, X_val_tr, y_val,
    n_repeats=10, random_state=SEED, scoring="roc_auc", n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": feature_names_ohe,
    "importance_mean": perm_imp.importances_mean,
    "importance_std": perm_imp.importances_std,
}).sort_values("importance_mean", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
top_perm = perm_df.head(15)
ax.barh(top_perm["feature"], top_perm["importance_mean"],
        xerr=top_perm["importance_std"], color="steelblue")
ax.set_xlabel("Mean AUC decrease")
ax.set_title(f"Permutation Importance ({diag_model_name}, val set)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.1b SHAP Values ──
if HAS_SHAP:
    # TreeExplainer is fast for tree-based models
    explainer = shap.TreeExplainer(diag_model)
    # Use a subsample for speed
    sample_idx = np.random.RandomState(SEED).choice(len(X_val_tr), size=min(500, len(X_val_tr)), replace=False)
    X_shap = X_val_tr[sample_idx]

    shap_values = explainer.shap_values(X_shap)
    # For binary classification, shap_values is a list [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    # Summary plot (beeswarm)
    print("SHAP Summary Plot (top 15 features):")
    shap.summary_plot(shap_vals, X_shap, feature_names=feature_names_ohe,
                      max_display=15, show=True)

    # Bar plot of mean |SHAP| per feature
    print("\nSHAP Bar Plot (mean |SHAP value|):")
    shap.summary_plot(shap_vals, X_shap, feature_names=feature_names_ohe,
                      plot_type="bar", max_display=15, show=True)
else:
    print("SHAP not installed. Install with: pip install shap")

### <a id="calibration"></a> 9.2 - Calibration

A model is **well-calibrated** if when it predicts 70% probability, roughly 70% of those cases are actually positive.
Good calibration matters when you use the probabilities for decisions (e.g., pricing, risk scoring).

📚 [sklearn Calibration Guide](https://scikit-learn.org/stable/modules/calibration.html) · [calibration_curve](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html) · Platt (1999), "Probabilistic Outputs for Support Vector Machines"

In [ ]:
# ── 9.2 Calibration Plots ──
fig, ax = plt.subplots(figsize=(8, 6))

for name in ["LogisticRegression", "RandomForest", "GBM_sklearn"]:
    if name not in MODEL_REGISTRY:
        continue
    prob = MODEL_REGISTRY[name]["y_prob_val"]
    # Clip for models that may output values outside [0,1]
    prob = np.clip(prob, 0, 1)
    fraction_pos, mean_pred = calibration_curve(y_val, prob, n_bins=10, strategy="uniform")
    brier = brier_score_loss(y_val, prob)
    ax.plot(mean_pred, fraction_pos, "s-", label=f"{name} (Brier={brier:.4f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Plot (Reliability Diagram)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Points above the diagonal: model underestimates probability (actual > predicted).")
print("Points below: model overestimates probability.")
print("\nIf calibration is poor, consider CalibratedClassifierCV (Platt scaling or isotonic regression).")

### <a id="gains-lift"></a> 9.3 - Cumulative Gains and Lift

**Cumulative gains**: "If I score the entire population and act on the top X%, how many positives do I capture?"
**Lift**: "How many times better than random is my model at each decile?"

These are essential for business decisions like: "We can only review 20% of applications -- which 20%?"

📚 Kuhn & Johnson (2013), *Applied Predictive Modeling* — widely used in direct marketing and credit scoring

In [ ]:
# ── 9.3 Cumulative Gains and Lift Curves ──

def cumulative_gains_lift(y_true, y_prob, n_bins=20):
    """Compute cumulative gains and lift by decile."""
    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    n = len(y_true)
    total_pos = y_true.sum()

    gains = []
    for i in range(1, n_bins + 1):
        cutoff = int(n * i / n_bins)
        captured = y_sorted[:cutoff].sum()
        pct_population = i / n_bins
        pct_captured = captured / total_pos
        lift = pct_captured / pct_population
        gains.append({
            "pct_population": pct_population,
            "pct_captured": pct_captured,
            "lift": lift,
        })
    return pd.DataFrame(gains)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name in ["LogisticRegression", "RandomForest", "GBM_sklearn"]:
    if name not in MODEL_REGISTRY:
        continue
    prob = MODEL_REGISTRY[name]["y_prob_val"]
    gl = cumulative_gains_lift(y_val, prob, n_bins=20)

    axes[0].plot(gl["pct_population"], gl["pct_captured"], "o-", label=name, markersize=4)
    axes[1].plot(gl["pct_population"], gl["lift"], "o-", label=name, markersize=4)

# Cumulative gains: random baseline
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
axes[0].set_xlabel("% of Population Scored")
axes[0].set_ylabel("% of Positives Captured")
axes[0].set_title("Cumulative Gains Curve")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Lift: baseline = 1
axes[1].axhline(y=1, color="k", linestyle="--", alpha=0.4, label="Random (lift=1)")
axes[1].set_xlabel("% of Population Scored")
axes[1].set_ylabel("Lift")
axes[1].set_title("Lift Curve")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Practical readout
best_prob = MODEL_REGISTRY[diag_model_name]["y_prob_val"]
gl_best = cumulative_gains_lift(y_val, best_prob, n_bins=10)
pct20 = gl_best.loc[gl_best["pct_population"] == 0.2, "pct_captured"].values
pct30 = gl_best.loc[gl_best["pct_population"] == 0.3, "pct_captured"].values
if len(pct20):
    print(f"\n{diag_model_name}: scoring the top 20% captures {pct20[0]*100:.1f}% of all positives.")
if len(pct30):
    print(f"{diag_model_name}: scoring the top 30% captures {pct30[0]*100:.1f}% of all positives.")

### <a id="error-analysis"></a> 9.4 - Error Analysis and Contrast Model

Standard error analysis: slice errors by feature values to find systematic failure modes.

**Contrast model**: a meta-model that tries to predict **whether the main model made an error**.
It can use features that were NOT in the original model (metadata, process columns, text fields).
The contrast model's feature importance tells you **what drives the errors**.

📚 Ribeiro et al. (2016), "Why Should I Trust You?: Explaining the Predictions of Any Classifier", *KDD*

In [ ]:
# ── 9.4a Error Slicing ──
# Attach predictions back to the val dataframe for slicing
val_analysis = val_df.copy()
val_analysis["y_true"] = y_val
val_analysis["y_pred"] = MODEL_REGISTRY[diag_model_name]["y_pred_val"]
val_analysis["y_prob"] = MODEL_REGISTRY[diag_model_name]["y_prob_val"]
val_analysis["is_error"] = (val_analysis["y_true"] != val_analysis["y_pred"]).astype(int)
val_analysis["error_type"] = "correct"
val_analysis.loc[(val_analysis["y_true"] == 1) & (val_analysis["y_pred"] == 0), "error_type"] = "false_negative"
val_analysis.loc[(val_analysis["y_true"] == 0) & (val_analysis["y_pred"] == 1), "error_type"] = "false_positive"

overall_error_rate = val_analysis["is_error"].mean()
print(f"Overall error rate: {overall_error_rate:.4f}")
print(f"\nError type distribution:\n{val_analysis['error_type'].value_counts()}")

# Slice by categorical features
for col in ["occupation", "workclass", "marital_status"]:
    if col not in val_analysis.columns:
        continue
    slice_err = (
        val_analysis.groupby(col)["is_error"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "error_rate", "count": "n"})
        .sort_values("error_rate", ascending=False)
    )
    print(f"\nError rate by {col} (top 5):")
    display(slice_err.head(5))

In [ ]:
# ── 9.4b Contrast Model ──
# The idea: train a model to predict is_error, using features that may NOT be in
# the main model (metadata, process columns, text-derived features, etc.).
# The contrast model's feature importance reveals what DRIVES the errors.

# Build contrast features: original features + metadata/process features
contrast_features = []
contrast_feature_names = []

# Include original numeric features
for c in numeric_cols:
    vals = to_numeric_loose(val_df[c]).fillna(0).values
    contrast_features.append(vals)
    contrast_feature_names.append(c)

# Include categorical features as label-encoded
for c in categorical_cols:
    le = LabelEncoder()
    # Fit on train, transform val (unseen -> -1 trick)
    train_cats = train_df[c].astype(str).fillna("MISSING")
    val_cats = val_df[c].astype(str).fillna("MISSING")
    le.fit(train_cats)
    val_encoded = val_cats.map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    contrast_features.append(val_encoded.values)
    contrast_feature_names.append(c)

# Include metadata/process columns that the main model does NOT use
extra_cols = ["db_etl_batch_id", "extract_country_code", "dataset_schema_version", "dgp_regime"]
for c in extra_cols:
    if c not in val_df.columns:
        continue
    le = LabelEncoder()
    train_vals = train_df[c].astype(str).fillna("MISSING")
    val_vals = val_df[c].astype(str).fillna("MISSING")
    le.fit(train_vals)
    encoded = val_vals.map(lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1)
    contrast_features.append(encoded.values)
    contrast_feature_names.append(f"META_{c}")

# Add the model's predicted probability as a feature
contrast_features.append(val_analysis["y_prob"].values)
contrast_feature_names.append("main_model_prob")

X_contrast = np.column_stack(contrast_features)
y_contrast = val_analysis["is_error"].values

print(f"Contrast model features: {len(contrast_feature_names)}")
print(f"Contrast target (error rate): {y_contrast.mean():.4f}")

In [ ]:
# ── 9.4c Train and Interpret the Contrast Model ──
# We use cross-validation on the val set to avoid overfitting the contrast model.

contrast_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, min_samples_leaf=20, random_state=SEED,
)

# CV on val set for honest contrast model evaluation
contrast_cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
contrast_scores = cross_val_score(
    contrast_model, X_contrast, y_contrast, cv=contrast_cv, scoring="roc_auc",
)
print(f"Contrast model CV AUC: {contrast_scores.mean():.4f} +/- {contrast_scores.std():.4f}")
print("(AUC > 0.5 means errors are not random -- there's a pattern the contrast model can exploit)\n")

# Fit on full val set for importance inspection
contrast_model.fit(X_contrast, y_contrast)

contrast_imp = pd.DataFrame({
    "feature": contrast_feature_names,
    "importance": contrast_model.feature_importances_,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
contrast_imp.head(15).plot.barh(x="feature", y="importance", ax=ax, legend=False, color="salmon")
ax.set_title("Contrast Model: What Drives Prediction Errors?")
ax.set_xlabel("Feature Importance (MDI)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Interpretation: features ranked high by the contrast model indicate")
print("subgroups or conditions where the main model systematically fails.")
print("META_ features highlight process/data artifacts that correlate with errors.")

> **Do It Yourself**
>
> 1. Generate SHAP summary plot for the best model.
> 2. Build a calibration plot and assess whether the model needs recalibration.
> 3. Compute cumulative gains: what % of positives do you capture in the top 30%?
> 4. Build a contrast model on the false positives and interpret what drives misclassification.
>
> Hint for contrast model: try using `error_type == "false_positive"` as the target instead of all errors.

---

## <a id="comparison"></a> Section 10 - Model Comparison Summary

In [ ]:
# ── Final comparison table ──
final_df = RESULTS_DF.copy()
final_df["overfit_gap"] = final_df["train_auc"] - final_df["val_auc"]

display(
    final_df[["model", "train_auc", "val_auc", "overfit_gap", "val_f1",
              "val_precision", "val_recall", "n_params", "fit_time_s"]]
    .sort_values("val_auc", ascending=False)
    .reset_index(drop=True)
    .style.format({
        "train_auc": "{:.4f}", "val_auc": "{:.4f}", "overfit_gap": "{:.4f}",
        "val_f1": "{:.4f}", "val_precision": "{:.4f}", "val_recall": "{:.4f}",
        "fit_time_s": "{:.2f}",
    })
    .background_gradient(subset=["val_auc"], cmap="Greens")
    .background_gradient(subset=["overfit_gap"], cmap="Reds")
)

In [ ]:
# ── Visual comparison ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: val AUC by model
plot_df = final_df.sort_values("val_auc", ascending=True)
colors = ["#e74c3c" if g > 0.05 else "#5fba7d" for g in plot_df["overfit_gap"]]
axes[0].barh(plot_df["model"], plot_df["val_auc"], color=colors)
axes[0].set_xlabel("Val AUC")
axes[0].set_title("Validation AUC by Model (red = overfit gap > 0.05)")
axes[0].set_xlim(0.5, 1.0)

# Scatter: val AUC vs fit time
axes[1].scatter(plot_df["fit_time_s"], plot_df["val_auc"], s=80, zorder=3)
for _, row in plot_df.iterrows():
    axes[1].annotate(row["model"], (row["fit_time_s"], row["val_auc"]),
                     fontsize=7, ha="left", va="bottom")
axes[1].set_xlabel("Fit Time (seconds)")
axes[1].set_ylabel("Val AUC")
axes[1].set_title("Performance vs Training Cost")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey takeaway: there is no single best model.")
print("The choice depends on: performance, interpretability, training cost, calibration needs, and deployment constraints.")

## <a id="regression-lab"></a> Section 11 (Stretch) - Mini-Lab: Regression with Ames Housing

If time permits, adapt the workflow to a **regression** task.

What changes:
- Target is continuous (`sale_price`), not binary.
- Metrics: RMSE, MAE, R-squared instead of AUC/F1.
- No threshold, no confusion matrix, no calibration in the same sense.
- SHAP and feature importance still apply.

### TODO (students)
1. Load Ames Housing, split, preprocess.
2. Train at least 2 regression models (e.g., Ridge, RandomForestRegressor).
3. Report RMSE and R-squared on the validation set.
4. Run SHAP on the best model.

In [ ]:
# ── Lightweight reference solution for Ames regression ──
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

ames = pd.read_csv("day1/generated/ames_housing_issues.csv")

# Drop rows without sale_price
ames = ames.dropna(subset=["sale_price"]).copy()

# Simple feature set (numeric only for brevity)
ames_num_cols = ["gr_liv_area", "lot_area", "year_built", "overall_qual",
                 "garage_area", "total_bsmt_sf"]
ames_num_cols = [c for c in ames_num_cols if c in ames.columns]

# Deduplicate by property_id (keep first)
ames = ames.drop_duplicates(subset=["property_id"], keep="first")

X_ames = ames[ames_num_cols].copy()
y_ames = ames["sale_price"].values

# Impute missing
imp = SimpleImputer(strategy="median")
X_ames_imp = imp.fit_transform(X_ames)

X_a_train, X_a_val, y_a_train, y_a_val = train_test_split(
    X_ames_imp, y_ames, test_size=0.2, random_state=SEED,
)

ames_results = []
for name, model in [
    ("Ridge", Ridge(alpha=1.0)),
    ("RF_Regressor", RandomForestRegressor(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)),
    ("GBM_Regressor", GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=SEED)),
]:
    model.fit(X_a_train, y_a_train)
    y_pred = model.predict(X_a_val)
    rmse = root_mean_squared_error(y_a_val, y_pred)
    mae = mean_absolute_error(y_a_val, y_pred)
    r2 = r2_score(y_a_val, y_pred)
    ames_results.append({"model": name, "RMSE": rmse, "MAE": mae, "R2": r2})
    print(f"[{name}]  RMSE={rmse:,.0f}  MAE={mae:,.0f}  R2={r2:.4f}")

show(pd.DataFrame(ames_results))

## <a id="checklist"></a> Section 12 - End-of-Notebook Checklist

> **Do It Yourself**
>
> Fill this checklist for your best model before considering it "ready":

> - [ ] Preprocessing is model-appropriate (scaling for linear/SVM/NN, encoding for all).
> - [ ] Splitting strategy matches data structure (group/time/stratified as needed).
> - [ ] Metrics chosen are appropriate for the problem and business context.
> - [ ] Overfitting checked (train vs val gap).
> - [ ] Cross-validation used for hyperparameter selection.
> - [ ] Pipeline ensures no preprocessing leakage inside CV.
> - [ ] Feature importance inspected with at least 2 methods.
> - [ ] Calibration assessed.
> - [ ] Error analysis performed (at least slicing, ideally contrast model).
> - [ ] Results are reproducible (fixed seeds, logged hyperparameters).

In [ ]:
reference_checklist = {
    "dataset": "adult_income_issues.csv",
    "preprocessing_model_appropriate": True,
    "split_strategy": "Stratified + entity-aware (person_id), using provided split column",
    "metrics_chosen": "AUC-ROC (primary), F1, Precision, Recall, Log-loss",
    "overfitting_checked": True,
    "cv_for_tuning": True,
    "pipeline_no_leakage": True,
    "feature_importance_2_methods": "Permutation importance + SHAP",
    "calibration_assessed": True,
    "error_analysis_done": "Slicing + Contrast model",
    "reproducible": True,
}

pd.DataFrame(reference_checklist, index=[0]).T.rename(columns={0: "Status"})

---

## References

### Models
- scikit-learn, [Supervised Learning Guide](https://scikit-learn.org/stable/supervised_learning.html)
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32.
- Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics*, 29(5), 1189–1232.
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD*.
- Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NeurIPS*.

### Evaluation & Model Selection
- scikit-learn, [Model Evaluation Guide](https://scikit-learn.org/stable/modules/model_evaluation.html)
- scikit-learn, [Cross-validation Guide](https://scikit-learn.org/stable/modules/cross_validation.html)
- Bergstra, J. & Bengio, Y. (2012). Random Search for Hyper-Parameter Optimization. *JMLR*, 13, 281–305.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. Ch. 7: Model Assessment and Selection.

### Interpretability & Diagnostics
- Lundberg, S. M. & Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions. *NeurIPS*.
- SHAP, [Documentation](https://shap.readthedocs.io/)
- Platt, J. (1999). Probabilistic Outputs for Support Vector Machines and Comparisons to Regularized Likelihood Methods.
- Ribeiro, M. T., Singh, S., & Guestrin, C. (2016). "Why Should I Trust You?": Explaining the Predictions of Any Classifier. *KDD*.
- Kuhn, M. & Johnson, K. (2013). *Applied Predictive Modeling*. Springer.

## <a id="acceptance"></a> Acceptance Checks

These checks validate the Day 2 modeling workflow end-to-end.

In [ ]:
# 1) Data integrity
assert X_train_sc.shape[0] == len(y_train)
assert X_val_sc.shape[0] == len(y_val)
assert np.isfinite(X_train_sc).all()
assert np.isfinite(X_train_tr).all()

# 2) Model registry: at least 5 model families trained
assert len(MODEL_REGISTRY) >= 5, f"Expected >= 5 models, got {len(MODEL_REGISTRY)}"

# 3) Metric consistency: all models evaluated with same metric set
assert all(col in RESULTS_DF.columns for col in ["train_auc", "val_auc", "val_f1"])

# 4) Overfitting check: at least one model shows train >> val gap
assert (RESULTS_DF["train_auc"] - RESULTS_DF["val_auc"]).max() > 0.01, "Expected at least one model with overfit gap"

# 5) Diagnostic outputs produced
assert len(perm_df) > 0, "Permutation importance not computed"
assert len(contrast_imp) > 0, "Contrast model importance not computed"

# 6) Reproducibility
assert SEED == 42

print("All acceptance checks passed.")
print("Notebook is ready for Day 2 delivery.")